# current season research for daily look

In [50]:
'''
Calculate the distance traveled and days on the road for each team over the entire season
schedule
'''
import pandas as pd
import numpy as np
import datetime

season_str = '2025-26'
#cutoff_date = datetime.datetime.now()
cutoff_date = pd.to_datetime('2025-11-26')

##### importing custom modules from the projects folder
import sys
from pathlib import Path
# Start at current working directory
current = Path.cwd()
# Walk up the tree until config.py is found or root is reached
for parent in [current] + list(current.parents):
    config_path = parent / "config.py"
    if config_path.exists():
        sys.path.append(str(parent))
        import config # <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<< 
        break
else:
    raise FileNotFoundError("config.py not found in any parent directories")

import scripts.functions.NBAteamSeasonTravel as travel


traveler = travel.seasonTeamTravelDistanceCalculator()
# access csv that has calculated distances from arena to arena
traveler.get_arena_distances()

# pull schedules for a season
traveler.get_season_schedule(
    season_str=season_str, 
    remove_gametypes = ['001', '003', '004', '005', '006']  # filters down to only regular season using ['001', '003', '004', '005', '006']. None or empty will provide full schedules
)
# process the data
traveler.process_schedule(cutoff_date = cutoff_date)
sched = traveler.schedules['processed']['2025-26']
sched.round(0)

,gameId,gameDate,team,opp,home,is_b2b,prev_home,next_home,days_rest,distance_miles,...,days_since_last_game,prev_opp,arena_from,arena_to,homeTeam_teamName,awayTeam_teamName,homeTeam_teamId,awayTeam_teamId,teamId,oppId
0,0022500082,2025-10-22,Hawks,Raptors,True,False,True,False,NaN,0.0,...,NaN,0,1610612737,1610612737,Hawks,Raptors,1610612737,1610612761,1610612737,1610612761
1,0022500090,2025-10-24,Hawks,Magic,False,False,True,True,1.0,402.0,...,2.0,1610612761,1610612737,1610612753,Magic,Hawks,1610612753,1610612737,1610612737,1610612753
2,0022500101,2025-10-25,Hawks,Thunder,True,True,False,False,0.0,402.0,...,1.0,1610612753,1610612753,1610612737,Hawks,Thunder,1610612737,1610612760,1610612737,1610612760
3,0022500115,2025-10-27,Hawks,Bulls,False,False,True,False,1.0,589.0,...,2.0,1610612760,1610612737,1610612741,Bulls,Hawks,1610612741,1610612737,1610612737,1610612741
4,0022500130,2025-10-29,Hawks,Nets,False,False,False,False,1.0,715.0,...,2.0,1610612741,1610612741,1610612751,Nets,Hawks,1610612751,1610612737,1610612737,1610612751
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
521,0022500229,2025-11-15,Hornets,Thunder,True,True,False,False,0.0,660.0,...,1.0,1610612749,1610612749,1610612766,Hornets,Thunder,1610612766,1610612760,1610612766,1610612760
522,0022500245,2025-11-17,Hornets,Raptors,False,False,True,False,1.0,587.0,...,2.0,1610612760,1610612766,1610612761,Raptors,Hornets,1610612761,1610612766,1610612766,1610612761
523,0022500256,2025-11-19,Hornets,Pacers,False,False,False,True,1.0,440.0,...,2.0,1610612761,1610612761,1610612754,Pacers,Hornets,1610612754,1610612766,1610612766,1610612754
524,0022500268,2025-11-22,Hornets,Clippers,True,False,False,False,2.0,428.0,...,3.0,1610612754,1610612754,1610612766,Hornets,Clippers,1610612766,1610612746,1610612766,1610612746


# historical research

In [ ]:
"""
It looks like the first season available in ScheduleLeagueV2 is 2017-18
"""

import pandas as pd
import numpy as np
import time

season_strs = [
    '2017-18', '2018-19', '2019-20', '2020-21', '2021-22', '2022-23', 
    '2023-24', '2024-25'
]

##### importing custom modules from the projects folder
import sys
from pathlib import Path
# Start at current working directory
current = Path.cwd()
# Walk up the tree until config.py is found or root is reached
for parent in [current] + list(current.parents):
    config_path = parent / "config.py"
    if config_path.exists():
        sys.path.append(str(parent))
        import config # <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<< 
        break
else:
    raise FileNotFoundError("config.py not found in any parent directories")

import scripts.functions.NBAteamSeasonTravel as travel
import scripts.functions.NBAhelperfunctions as hf

traveler = travel.seasonTeamTravelDistanceCalculator()
# access csv that has calculated distances from arena to arena
traveler.get_arena_distances()

# pull in requested seasons
for i in season_strs:

    traveler.get_season_schedule(
        season_str=i, 
        remove_gametypes = ['001', '003', '004', '005', '006']  # filters down to only regular season using ['001', '003', '004', '005', '006']. None or empty will provide full schedules
    )
    time.sleep(2)
    print(i, 'retrieved...')

# process the data - adds travel calcs, days rest, road trip calcs
# as well as odds for data
traveler.process_historical_schedules()
df = traveler.schedules['historical']



2017-18 retrieved...
2018-19 retrieved...
2019-20 retrieved...
2020-21 retrieved...
2021-22 retrieved...
2022-23 retrieved...
2023-24 retrieved...
2024-25 retrieved...


In [32]:
# remove games that don't have odds data
df = df[df['total'].notna()]
df.shape

(12986, 48)

In [33]:
# only away games are kept since this is looking at road trip performance
d = df[~df['home']]
d.shape

(6493, 48)

In [49]:
d.loc[:,'homeFav'] = d['homeSpread'] < 0

d.loc[:,'totalScored'] = d['homeTeam_score'] + d['awayTeam_score']
d.loc[:,'overTotal'] = d['totalScored'] > d['total']
d.loc[:,'underTotal'] = ~d['overTotal']

d.loc[:,'awayDelta'] = d['awayTeam_score'] - d['homeTeam_score']
d.loc[:,'awayAts'] = d['awayDelta'] + d['awaySpread']

d.loc[:,'homeImpliedWinProb'] = hf.convert_ameri_odds_to_probability(d['homeMoneyline'])
d.loc[:,'awayImpliedWinProb'] = hf.convert_ameri_odds_to_probability(d['awayMoneyline'])
d.loc[:,'impTotalProb'] = d['homeImpliedWinProb'] + d['awayImpliedWinProb']

In [44]:
d

,gameId,gameDate,seasonYear,weekNumber,day,monthNum,win,team,opp,homeTeam_score,...,awaySpread,awayMoneyline,totalScored,overTotal,underTotal,awayDelta,awayAts,homeImpliedWinProb,awayImpliedWinProb,impTotalProb
0,0021700009,2017-10-18,2017-18,1,Wed,10,True,Hawks,Mavericks,111,...,5.5,190.0,228,True,False,6,11.5,0.705882,0.344828,1.050710
1,0021700017,2017-10-20,2017-18,1,Fri,10,False,Hawks,Hornets,109,...,4.5,160.0,200,False,True,-18,-13.5,0.655172,0.384615,1.039788
2,0021700038,2017-10-22,2017-18,1,Sun,10,False,Hawks,Nets,116,...,1.0,-105.0,220,False,True,-12,-11.0,0.534884,0.512195,1.047079
3,0021700042,2017-10-23,2017-18,2,Mon,10,False,Hawks,Heat,104,...,10.5,500.0,197,False,True,-11,-0.5,0.875000,0.166667,1.041667
4,0021700065,2017-10-26,2017-18,2,Thu,10,False,Hawks,Bulls,91,...,2.0,110.0,177,False,True,-5,-3.0,0.565217,0.476190,1.041408
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14070,0022200513,2022-12-27,2022-23,11,Tue,12,False,Hornets,Warriors,110,...,5.0,170.0,215,False,True,-5,0.0,0.655172,0.370370,1.025543
14075,0022200583,2023-01-06,2022-23,12,Fri,1,True,Hornets,Bucks,109,...,10.0,380.0,247,True,False,29,39.0,0.821429,0.208333,1.029762
14076,0022200598,2023-01-08,2022-23,12,Sun,1,False,Hornets,Pacers,116,...,5.5,190.0,227,False,True,-5,0.5,0.687500,0.344828,1.032328
14077,0022200613,2023-01-10,2022-23,13,Tue,1,False,Hornets,Raptors,132,...,7.5,250.0,252,True,False,-12,-4.5,0.750000,0.285714,1.035714
